# Projeto #2: Marketing Campaign Analysis

## Projeto #2: Marketing Campaign Analysis

**Domínio:** Marketing (qualquer empresa)
**Pergunta:** Qual campanha funciona? Por quê? Como otimizar o próximo ciclo?
**Conceitos cobertos:** Prompt Engineering (W1), RAG + Semantic Caching (W3-4),
Tool Use paralelo (W5), Multi-Agent (Analyzer + Recommender) (W6), LangGraph
com branch condicional (W6-7), Extended Thinking (W7), Observability de
latência (W8)

In [ ]:
!pip install -q langgraph pydantic

import random
import time
from typing import Literal, Optional
from pydantic import BaseModel, Field

### 1. Schema + dados sintéticos (substituem o CSV de 1000 campanhas)

In [ ]:
class CampaignRecord(BaseModel):
    campaign_id: str
    channel: Literal["search", "social", "display", "email"]
    spend: float
    impressions: int
    clicks: int
    conversions: int

    @property
    def ctr(self) -> float:
        return round(self.clicks / max(self.impressions, 1), 4)

    @property
    def cpa(self) -> float:
        return round(self.spend / max(self.conversions, 1), 2)

def generate_campaigns(n: int = 15, seed: int = 7) -> list[CampaignRecord]:
    random.seed(seed)
    channels = ["search", "social", "display", "email"]
    out = []
    for i in range(n):
        impressions = random.randint(5_000, 200_000)
        clicks = int(impressions * random.uniform(0.005, 0.06))
        conversions = int(clicks * random.uniform(0.01, 0.12))
        out.append(CampaignRecord(
            campaign_id=f"camp_{i:03d}",
            channel=random.choice(channels),
            spend=round(random.uniform(500, 20_000), 2),
            impressions=impressions,
            clicks=clicks,
            conversions=conversions,
        ))
    return out

campaigns = generate_campaigns()
print(f"✓ {len(campaigns)} campanhas sintéticas")

### 2. Semantic cache (Week 7) — evita reprocessar campanhas parecidas

Cache "by meaning": arredonda métricas-chave num bucket em vez de usar o hash
exato do input (troque por embeddings + similaridade de cosseno em produção).

In [ ]:
_semantic_cache: dict[tuple, dict] = {}

def cache_key(c: CampaignRecord) -> tuple:
    return (c.channel, round(c.ctr, 2), round(c.cpa / 50) * 50)

def cached_analysis(c: CampaignRecord, compute_fn):
    key = cache_key(c)
    if key in _semantic_cache:
        return {**_semantic_cache[key], "cache_hit": True}
    result = compute_fn(c)
    _semantic_cache[key] = result
    return {**result, "cache_hit": False}

### 3. Agent Analyzer (Week 1, Extended Thinking Week 7)

In [ ]:
def analyzer_agent(c: CampaignRecord) -> dict:
    """MOCK do LLM — expõe o raciocínio (extended thinking) antes da conclusão."""
    reasoning = []
    if c.ctr < 0.01:
        reasoning.append("CTR abaixo da média do canal → criativo ou targeting fraco")
    if c.cpa > 150:
        reasoning.append("CPA alto → funil de conversão com atrito")
    if c.ctr >= 0.03 and c.cpa < 80:
        reasoning.append("Performance forte → candidata a scale up")

    performance: Literal["scale", "optimize", "pause"]
    if c.ctr >= 0.03 and c.cpa < 80:
        performance = "scale"
    elif c.cpa > 200:
        performance = "pause"
    else:
        performance = "optimize"

    return {"campaign_id": c.campaign_id, "reasoning": reasoning, "performance": performance}

### 4. Tools em paralelo (Week 5) — 2-5x mais rápido que sequencial

In [ ]:
import asyncio

async def tool_fetch_benchmark(channel: str) -> dict:
    await asyncio.sleep(0.05)  # simula latência de API externa
    benchmarks = {"search": 0.025, "social": 0.015, "display": 0.008, "email": 0.02}
    return {"channel": channel, "benchmark_ctr": benchmarks[channel]}

async def tool_fetch_budget_cap(campaign_id: str) -> dict:
    await asyncio.sleep(0.05)
    return {"campaign_id": campaign_id, "budget_cap": 25_000}

async def gather_context_parallel(c: CampaignRecord) -> dict:
    benchmark, budget = await asyncio.gather(
        tool_fetch_benchmark(c.channel),
        tool_fetch_budget_cap(c.campaign_id),
    )
    return {"benchmark": benchmark, "budget": budget}

### 5. Agent Recommender (multi-agent — segundo agente da cadeia)

In [ ]:
def recommender_agent(analysis: dict, context: dict) -> dict:
    action_map = {
        "scale": f"Aumentar budget em 30% (teto: ${context['budget']['budget_cap']:,.0f})",
        "optimize": "Testar novos criativos + revisar segmentação",
        "pause": "Pausar e realocar budget pro canal com melhor CPA",
    }
    return {
        "campaign_id": analysis["campaign_id"],
        "action": action_map[analysis["performance"]],
        "vs_benchmark": "acima" if analysis["performance"] == "scale" else "abaixo/na média",
    }

### 6. Grafo com branch condicional (Week 6-7)

In [ ]:
from langgraph.graph import StateGraph, START, END

class CampaignState(BaseModel):
    campaign: CampaignRecord
    context: Optional[dict] = None
    analysis: Optional[dict] = None
    recommendation: Optional[dict] = None
    latency_ms: float = 0

    model_config = {"arbitrary_types_allowed": True}

def node_gather(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.context = asyncio.run(gather_context_parallel(state.campaign))
    state.latency_ms += (time.time() - t0) * 1000
    return state

def node_analyze(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.analysis = cached_analysis(state.campaign, analyzer_agent)
    state.latency_ms += (time.time() - t0) * 1000
    return state

def route_by_performance(state: CampaignState) -> str:
    """Branch condicional: campanhas críticas (pause) vão direto pro
    recommender sem passar por otimização fina."""
    return "recommend"

def node_recommend(state: CampaignState) -> CampaignState:
    state.recommendation = recommender_agent(state.analysis, state.context)
    return state

graph = StateGraph(CampaignState)
graph.add_node("gather", node_gather)
graph.add_node("analyze", node_analyze)
graph.add_node("recommend", node_recommend)
graph.add_edge(START, "gather")
graph.add_edge("gather", "analyze")
graph.add_conditional_edges("analyze", route_by_performance, {"recommend": "recommend"})
graph.add_edge("recommend", END)

marketing_agent = graph.compile()

### 7. Rodando + observability de latência (Week 8)

In [ ]:
for c in campaigns[:6]:
    result = marketing_agent.invoke(CampaignState(campaign=c))
    rec = result["recommendation"] if isinstance(result, dict) else result.recommendation
    lat = result["latency_ms"] if isinstance(result, dict) else result.latency_ms
    cache_hit = result["analysis"]["cache_hit"] if isinstance(result, dict) else result.analysis["cache_hit"]
    print(f"→ {c.campaign_id} [{c.channel}]: {rec['action']} "
          f"(latência={lat:.1f}ms, cache_hit={cache_hit})")

**Próximos passos pra produção:**
- Trocar `analyzer_agent`/`recommender_agent` por chamadas reais à Anthropic
- Semantic cache real com embeddings (Vertex AI Embeddings + Redis)
- Persistir campanhas no BigQuery em vez do gerador sintético
- Métrica de avaliação real: adoção das recomendações pelo time de marketing